In [8]:
import numpy as np
import pandas as pd
import time
import scipy.stats as spst
import scipy.special as spsp
import copy 
#import mp math as mp

import sys
sys.path.insert(sys.path.index("")+1, "C:/Users/27261/Desktop/3_Courses in PHBS/3_09_AppliedStochasticProcess/Project_sv32_EMC")
import pyfeng as pf
import pyfeng.ex as pfex
np.set_printoptions(precision=4)
from utils import *
seed_everything()
case_names = case_dict.keys()

全局随机数种子已锁定为: 123456


In [5]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [11]:
# 带跳3/2模型核心实现（继承纯扩散3/2，新增跳参数+跳动力学+鞅性校验+带跳特征函数+期权定价）

# 测试代码
if __name__ == "__main__":
    # 模型参数（参考论文2节的3/2模型参数，新增跳参数）
    sigma = 0.2450  # 初始波动率（论文2节：V0=0.2450²）
    vov = 8.56      # 方差波动率ε（论文2节）
    mr = 22.84      # 均值回归速度κ（论文2节）
    rho = -0.99     # 价-方差相关（论文2节）
    theta = 0.4669**2  # 长期方差（论文2节）
    # 跳参数（参考论文4节校准结果：λ=0.18, μ=-0.30, σ=0.39）
    lam = 0.18
    mu_j = -0.30
    sigma_j = 0.39
    # 市场参数
    intr = 0.02
    divr = 0.01

    # 初始化带跳3/2模型
    model = pfex.Sv32JumpMc(
        sigma=sigma, vov=vov, mr=mr, rho=rho, theta=theta,
        lam=lam, mu_j=mu_j, sigma_j=sigma_j,
        intr=intr, divr=divr, is_fwd=False
    )
    # 设置蒙特卡洛参数
    model.set_num_params(n_path=10000, dt=1/252, rn_seed=42)

    # 1. 定价股票欧式认购期权
    S0 = 100  # 标的初始价格
    K_stock = 100  # 股票期权执行价
    texp_stock = 0.25  # 3个月到期
    call_price = model.price_stock_option_cosine(S0=S0, K=K_stock, texp=texp_stock, is_call=True)
    print(f"股票认购期权价格：{call_price:.4f}")

鞅性条件校验通过：κ-ερ=31.314400 ≥ -ε²/2=-36.636800
股票认购期权价格：0.9954


In [7]:
import numpy as np
import scipy.special as spsp
from scipy.stats import ncx2, poisson, norm
from scipy.optimize import brentq
import warnings
import mpmath

warnings.filterwarnings("ignore")

class Sv32PlusJumpsExactSim:
    """
    Exact simulation of the 3/2 plus Jumps model based on conditional Fourier-Cosine (COS) method.
    References: 
      - Brignone & Junike (2026), European Journal of Operational Research
      - Baldeaux & Badran (2012), Consistent Modeling of VIX and Equity Derivatives
    """
    def __init__(self, spot, r, kappa, theta, epsilon, rho, v0, lam, mu_j, sigma_j):
        self.S0 = spot
        self.r = r
        # 3/2 model parameters
        self.kappa = kappa
        self.theta = theta
        self.epsilon = epsilon
        self.rho = rho
        self.v0 = v0
        # Jump parameters
        self.lam = lam
        self.mu_j = mu_j
        self.sigma_j = sigma_j
        
        # Calculate Martingale compensator for jumps
        self.m_bar = np.exp(self.mu_j + 0.5 * self.sigma_j**2) - 1.0
        
        # Map 3/2 parameters to CIR parameters for Y_t = 1/V_t
        self.k_Y = self.kappa * self.theta
        self.theta_Y = (self.kappa + self.epsilon**2) / (self.kappa * self.theta)
        self.sigma_Y = -self.epsilon
        self.Y0 = 1.0 / self.v0

        # COS Method Hyperparameters (Brignone & Junike 2026)
        self.n_x = 2**8
        self.L = 12.0  # Integration truncation domain [-L, L]

    def simulate_variance(self, T, n_paths):
        """Simulate terminal variance V_T using Non-central Chi-squared distribution"""
        c = (self.sigma_Y**2 * (np.exp(self.k_Y * T) - 1.0)) / (4.0 * self.k_Y)
        delta = 4.0 * self.k_Y * self.theta_Y / self.sigma_Y**2
        non_centrality = (self.Y0 * np.exp(self.k_Y * T)) / c
        
        # Simulate Y_T ~ 1/c * \chi^2(delta, non_centrality)
        chi2_samples = ncx2.rvs(delta, non_centrality, size=n_paths)
        Y_T = chi2_samples * c
        V_T = 1.0 / Y_T
        return Y_T, V_T

    def conditional_cf(self, u, Y_T, T):
        """
        Conditional Characteristic Function of diffusion log-return X_T given Y_T.
        Mapped from the 4/2 model framework in Brignone & Junike (2026).
        """
        # Intermediate constants based on Source 1 mapping (a=0, b=1)
        xi_1 = self.r * T - (self.rho * self.k_Y / self.sigma_Y) * np.log(self.Y0) + \
               (self.rho * self.k_Y * self.theta_Y * T / self.sigma_Y)
        xi_3 = self.rho / self.sigma_Y
        xi_5 = (self.rho * self.k_Y / self.sigma_Y) - 0.5
        xi_8 = 1.0 - self.rho**2
        
        # The argument u_2 for L(0, u_2)
        u_2 = -1j * u * xi_5 + 0.5 * (u**2) * xi_8
        
        # Bessel function index and arguments
        nu = (2.0 * self.k_Y * self.theta_Y / self.sigma_Y**2) - 1.0
        nu_u2 = np.sqrt(nu**2 + 8.0 * u_2 / self.sigma_Y**2)
        
        z = 2.0 * self.k_Y * np.sqrt(self.Y0 * Y_T) / (self.sigma_Y**2 * np.sinh(self.k_Y * T / 2.0))
        
        # --- 修复核心：使用 mpmath 处理复数阶贝塞尔函数并安全计算比值 ---
        L_val = np.zeros_like(nu_u2, dtype=complex)
        
        # 标量：实数阶的分母可以直接先算出来 (使用 mpmath.besseli)
        bessel_den_mp = mpmath.besseli(nu, z)
        
        for k in range(len(nu_u2)):
            # 数组遍历：复数阶的分子
            bessel_num_mp = mpmath.besseli(nu_u2[k], z)
            
            # 直接在 mpmath 的高精度类型下相除，规避 float64 溢出，然后强转回原生 complex
            # 即使分子分母达到 10^500，它们的比值通常是安全的浮点数范围
            L_val[k] = complex(bessel_num_mp / bessel_den_mp)
        # -----------------------------------------------------------------
        
        # Full conditional CF
        cf = np.exp(1j * u * (xi_1 + xi_3 * np.log(Y_T))) * L_val
        return cf

    def construct_conditional_cdf_cos(self, Y_T_path, T):
        """
        Constructs the inverse CDF function for a specific Y_T path using the COS method.
        """
        a = -self.L
        b = self.L
        k_vec = np.arange(self.n_x)
        omega = k_vec * np.pi / (b - a)
        
        cf_vals = self.conditional_cf(omega, Y_T_path, T)
        real_term = (cf_vals * np.exp(-1j * omega * a)).real
        real_term[0] *= 0.5
        
        def cdf_func(y):
            # Equation (8) from Brignone & Junike (2026)
            V_k = np.zeros(self.n_x)
            V_k[0] = np.minimum(y, b) - a
            
            idx = k_vec[1:]
            V_k[1:] = (2.0 * (b - a) / (np.pi * idx)) * np.sin(idx * np.pi * (np.minimum(y, b) - a) / (b - a))
            
            # Series summation
            cdf_val = np.sum(real_term * V_k) / (b - a)
            return cdf_val
            
        return cdf_func

    def exact_simulation_paths(self, T, n_paths):
        """
        Generate exact paths for the 3/2 plus jumps model.
        """
        # Step 1: Simulate Terminal Variance V_T and Y_T
        Y_T_paths, V_T_paths = self.simulate_variance(T, n_paths)
        
        log_returns = np.zeros(n_paths)
        
        for i in range(n_paths):
            # Step 2: Continuous Diffusion Part via Conditional COS Root-finding
            cdf_func = self.construct_conditional_cdf_cos(Y_T_paths[i], T)
            u_rand = np.random.uniform(0, 1)
            
            # Root finding to invert CDF: P(X_T <= y | V_T) = U
            try:
                # Using Brent's method. Initial interval is [-L, L]
                x_diff = brentq(lambda y: cdf_func(y) - u_rand, -self.L, self.L)
            except ValueError:
                # Fallback if probability mass escapes [-L, L]
                x_diff = 0.0 
                
            # Step 3: Simulate independent Jumps
            n_jumps = poisson.rvs(self.lam * T)
            if n_jumps > 0:
                jump_sum = np.sum(norm.rvs(loc=self.mu_j, scale=self.sigma_j, size=n_jumps))
            else:
                jump_sum = 0.0
                
            # Jump compensator adjustment
            jump_comp = -self.lam * self.m_bar * T
            
            # Step 4: Combine
            log_returns[i] = x_diff + jump_sum + jump_comp
            
        S_T = self.S0 * np.exp(log_returns)
        return S_T, V_T_paths

# Example Usage
if __name__ == "__main__":
    # Parameters from Baldeaux & Badran (2012)
    model = Sv32PlusJumpsExactSim(
        spot=100.0, r=0.0, 
        kappa=30.84, theta=0.10**2, epsilon=50.56, rho=-0.57, v0=0.0822**2,
        lam=0.18, mu_j=-0.30, sigma_j=0.39
    )
    
    T = 9.0 / 365.0  # 9 days
    n_paths = 1000
    
    # Run exact simulation
    S_T, V_T = model.exact_simulation_paths(T, n_paths)
    print(f"Mean Expected S_T (Martingale Test): {np.mean(S_T):.4f} (Should be close to 100)")

Mean Expected S_T (Martingale Test): 158.0714 (Should be close to 100)
